In [ ]:
!gdown 1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I

Downloading...
From: https://drive.google.com/uc?id=1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I
To: /content/df_train.csv
100% 286k/286k [00:00<00:00, 16.6MB/s]


In [ ]:
!gdown 1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG

Downloading...
From: https://drive.google.com/uc?id=1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG
To: /content/df_test.csv
100% 71.9k/71.9k [00:00<00:00, 74.5MB/s]


In [ ]:
!nvidia-smi

Thu Jun 11 04:40:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             29W /   70W |     391MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd

df = pd.read_csv("df_train.csv")
df

,dwt_mean,dwt_max,dwt_min,dwt_median,dwt_q1,dwt_q3,dwt_iqr,mfcc_mean,mfcc_std,mfcc_max,...,rms_min,rms_median,rms_var,rms_skew,rms_kurtosis,rms_q1,rms_q3,rms_range,shannon_entropy,audio_type
0,1.206750e-08,0.013878,-0.014668,-1.500644e-11,-0.000028,0.000026,0.000055,-81.473040,295.34805,152.37076,...,9.717603e-08,0.000048,1.730892e-07,1.283449,0.308629,1.093666e-05,0.000524,0.001483,8.866710,myocardial
1,5.968118e-07,0.039521,-0.040875,1.670743e-11,-0.000016,0.000017,0.000033,-82.751785,298.57016,158.75189,...,1.647872e-07,0.000034,7.909553e-08,4.731361,22.492230,9.232333e-06,0.000076,0.001755,5.972518,myocardial
2,3.870791e-07,0.032473,-0.023627,7.594341e-12,-0.000017,0.000016,0.000033,-81.980400,297.24650,171.03497,...,1.124997e-07,0.000082,3.880997e-08,3.890662,20.913025,3.342536e-06,0.000215,0.001343,7.658798,myocardial
3,-4.941957e-07,0.131609,-0.118817,-2.006760e-11,-0.000014,0.000014,0.000028,-77.834270,280.35214,168.92433,...,9.554750e-08,0.000026,5.670202e-06,4.185934,17.443203,5.695775e-06,0.000075,0.015121,7.720282,myocardial
4,4.204849e-07,0.002665,-0.002124,-7.600626e-11,-0.000031,0.000029,0.000060,-82.015420,298.52830,97.61485,...,8.227275e-07,0.000036,3.165749e-09,0.953618,-0.323343,1.287166e-05,0.000091,0.000202,9.517833,myocardial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,-1.585129e-08,0.011102,-0.009699,-2.031497e-09,-0.000019,0.000020,0.000039,-82.583570,297.38315,139.96802,...,1.878590e-06,0.000031,8.288504e-09,3.186870,11.402454,1.837468e-05,0.000060,0.000525,7.771392,normal
444,5.693161e-07,0.022825,-0.028649,-4.278836e-11,-0.000013,0.000013,0.000027,-83.153275,299.87430,200.29291,...,1.362070e-07,0.000027,1.346874e-07,3.889586,15.481110,1.185905e-05,0.000067,0.002202,5.310026,normal
445,4.123294e-09,0.060122,-0.060773,-4.331842e-10,-0.000004,0.000004,0.000008,-84.163050,296.94974,180.51411,...,7.595175e-07,0.000011,9.946041e-07,2.224082,3.260276,5.449809e-06,0.000059,0.003769,5.532661,normal
446,-3.339084e-08,0.089756,-0.102157,1.016094e-11,-0.000002,0.000002,0.000004,-82.164860,292.40936,203.16748,...,1.403314e-07,0.000011,1.463136e-06,5.434349,30.395962,8.566282e-07,0.000142,0.008902,6.255238,normal


In [ ]:
import numpy as np
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, MaxPooling1D, LSTM, Dense, Dropout, Flatten

In [ ]:
model = Sequential()

input_shape = (51, 1)
num_classes = 2

model.add(Conv1D(filters=16, kernel_size=3, padding='same', input_shape=input_shape))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(Conv1D(filters=16, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(LSTM(32, return_sequences=True))

model.add(LSTM(128, return_sequences=False))

model.add(Dense(64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))

model.add(Dense(num_classes, activation='softmax'))

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_5 (Conv1D)               │ (None, 51, 16)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 51, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 51, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 25, 16)         │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 25, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 12, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 12, 32)         │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,322 (384.07 KB)

 Trainable params: 98,130 (383.32 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np

# 1. Data Splitting (Source: 131)
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

# 2. 10-Fold Cross-Validation (Source: 272)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
# Initialize lists to store metrics per fold
fold_metrics = {
    'accuracy': [],
    'sensitivity': [],
    'specificity': [],
    'auroc': []
}

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    # Re-initialize or fit model
    model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=0,
        validation_data=(X_fold_val, y_fold_val)
    )

    # Generate predictions for the validation fold
    y_pred_probs = model.predict(X_fold_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_fold_val, axis=1)

    # Calculate Metrics
    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Sensitivity & Specificity
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    # AUROC (using probabilities for the positive class)
    auroc = roc_auc_score(y_fold_val[:, 1], y_pred_probs[:, 1])

    # Store results
    fold_metrics['accuracy'].append(acc)
    fold_metrics['sensitivity'].append(sens)
    fold_metrics['specificity'].append(spec)
    fold_metrics['auroc'].append(auroc)

    print(f'Fold {fold_no} Results: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}')
    fold_no += 1

# Summary
print(f'\n------------------------------------------------------------------------')
print(f'Summary Results (Mean ± STD) over 10 Folds:')
for metric, values in fold_metrics.items():
    print(f'{metric.capitalize()}: {np.mean(values)*100:.2f}% ± {np.std(values)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Fold 1 Results: Acc=0.7561, Sens=0.4444, Spec=1.0000, AUROC=0.8140
------------------------------------------------------------------------
Training for Fold 2 ...
Fold 2 Results: Acc=0.6585, Sens=1.0000, Spec=0.4400, AUROC=0.9100
------------------------------------------------------------------------
Training for Fold 3 ...
Fold 3 Results: Acc=0.8780, Sens=0.8500, Spec=0.9048, AUROC=0.9524
------------------------------------------------------------------------
Training for Fold 4 ...
Fold 4 Results: Acc=0.8000, Sens=0.9565, Spec=0.5882, AUROC=0.9412
------------------------------------------------------------------------
Training for Fold 5 ...
Fold 5 Results: Acc=0.9250, Sens=1.0000, Spec=0.8421, AUROC=0.9850
------------------------------------------------------------------------
Training for Fold 6 ...
Fold 6 Results: Acc=0.9500, Se

In [ ]:
import numpy as np
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D, Dense, Dropout, Flatten, LeakyReLU, Input

LEAKY_ALPHA = 0.01

model = Sequential()

input_shape = (51, 1)
num_classes = 2

model.add(Input(shape=input_shape))

model.add(Conv1D(filters=32, kernel_size=1, padding='same'))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))

model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(Conv1D(filters=32, kernel_size=3, padding='same', groups=32))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))

model.add(Conv1D(filters=64, kernel_size=1, padding='same'))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))

model.add(Flatten())

model.add(Dense(64))
model.add(BatchNormalization())
model.add(LeakyReLU(alpha=LEAKY_ALPHA))
model.add(Dropout(0.5))

model.add(Dense(num_classes, activation='softmax'))

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_7 (Conv1D)               │ (None, 51, 32)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 51, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 51, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 25, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 25, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 25, 64)         │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 25, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_6 (LeakyReLU)       │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_7 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,666 (412.76 KB)

 Trainable params: 105,282 (411.26 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
    # Option 1: Impute NaNs (e.g., with mean, median, or zero)
    # from sklearn.impute import SimpleImputer
    # imputer = SimpleImputer(strategy='mean')
    # X = imputer.fit_transform(X)
    # print("NaNs have been imputed with the mean.")

    # For now, we will proceed to see if other issues arise, but imputation is recommended.
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np

# 1. Data Splitting (Source: 131)
# "Training data... 70%, test data... 30%"
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
# Initialize lists to store metrics per fold
fold_metrics = {
    'accuracy': [],
    'sensitivity': [],
    'specificity': [],
    'auroc': []
}

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    # Training Parameters
    # Epochs: 30, Batch Size: 32
    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=0,
        validation_data=(X_fold_val, y_fold_val)
    )

    # Generate predictions for the validation fold
    y_pred_probs = model.predict(X_fold_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_fold_val, axis=1)

    # Calculate Metrics
    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Sensitivity & Specificity
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    auroc = roc_auc_score(y_fold_val[:, 1], y_pred_probs[:, 1])

    # Store results
    fold_metrics['accuracy'].append(acc)
    fold_metrics['sensitivity'].append(sens)
    fold_metrics['specificity'].append(spec)
    fold_metrics['auroc'].append(auroc)

    print(f'Fold {fold_no} Results: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}')
    fold_no += 1

# Summary
print(f'\n------------------------------------------------------------------------')
print(f'Summary Results (Mean ± STD) over 10 Folds:')
for metric, values in fold_metrics.items():
    print(f'{metric.capitalize()}: {np.mean(values)*100:.2f}% ± {np.std(values)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Fold 1 Results: Acc=0.7317, Sens=0.5000, Spec=0.9130, AUROC=0.8068
------------------------------------------------------------------------
Training for Fold 2 ...
Fold 2 Results: Acc=0.9024, Sens=0.8125, Spec=0.9600, AUROC=0.9875
------------------------------------------------------------------------
Training for Fold 3 ...
Fold 3 Results: Acc=0.9756, Sens=1.0000, Spec=0.9524, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 4 ...
Fold 4 Results: Acc=0.9750, Sens=1.0000, Spec=0.9412, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 5 ...
Fold 5 Results: Acc=1.0000, Sens=1.0000, Spec=1.0000, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 6 ...
Fold 6 Results: Acc=0.9750, Se

In [ ]:
"""
Creates the VGG-16 architecture modified for 1D PCG features.

Structure (Source: 459-462, Figure 12):
- 13 Convolutional Layers (grouped in 5 blocks)
- ReLU Activation
- 5 Max Pooling Layers
- 3 Fully Connected Layers
"""
model = Sequential()

input_shape=(51, 1)
num_classes=2

# --- Block 1 ---
# 2 Convolutional Layers + Max Pooling
model.add(Conv1D(64, kernel_size=3, padding='same', input_shape=input_shape))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(64, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

# --- Block 2 ---
# 2 Convolutional Layers + Max Pooling
model.add(Conv1D(128, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(128, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

# --- Block 3 ---
# 3 Convolutional Layers + Max Pooling
model.add(Conv1D(256, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(256, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(256, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

# --- Block 4 ---
# 3 Convolutional Layers + Max Pooling
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

# --- Block 5 ---
# 3 Convolutional Layers + Max Pooling
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv1D(512, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

# --- Classifier (Fully Connected) ---
model.add(Flatten())

model.add(Dense(4096))
model.add(BatchNormalization())
model.add(Activation('relu'))

model.add(Dense(4096))
model.add(BatchNormalization())
model.add(Activation('relu'))

model.add(Dense(num_classes))
model.add(Activation('softmax'))

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 51, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 51, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 51, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 51, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 51, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 51, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 25, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 25, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 25, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 12, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 12, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 12, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 12, 256)        │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 12, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 12, 256)        │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 12, 256)        │         1,02

 Total params: 23,847,746 (90.97 MB)

 Trainable params: 23,822,914 (90.88 MB)

 Non-trainable params: 24,832 (97.00 KB)

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
# 1. Data Splitting (Source: 131)
# "Training data... 70%, test data... 30%"
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
scores = []
val_scores = []

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    history = model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=1,
        validation_data=(X_fold_val, y_fold_val)
    )

    scores.append(history.history['accuracy'][-1])
    val_scores.append(history.history['val_accuracy'][-1])
    fold_no += 1

print(f'Average Training Accuracy across 10 folds: {np.mean(scores)*100:.2f}%')
print(f'Average Validation Accuracy across 10 folds: {np.mean(val_scores)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Epoch 1/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 1.0000 - loss: 2.6061e-04 - val_accuracy: 1.0000 - val_loss: 1.3584e-05
Epoch 2/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 1.0000 - loss: 1.2207e-04 - val_accuracy: 1.0000 - val_loss: 1.3380e-05
Epoch 3/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 1.1717e-04 - val_accuracy: 1.0000 - val_loss: 1.2555e-05
Epoch 4/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 3.9797e-05 - val_accuracy: 1.0000 - val_loss: 1.1784e-05
Epoch 5/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 3.1921e-04 - val_accuracy: 1.0000 - val_loss: 1.0714e-05
Epoch 6/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 3.8150e-05 - val_accuracy: 1.0000 - val_loss: 9.8652e-06
Epoch 7/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np

# 1. Data Splitting (Source: 131)
# "Training data... 70%, test data... 30%"
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
# Initialize dictionary to store metrics per fold
fold_metrics = {
    'accuracy': [],
    'sensitivity': [],
    'specificity': [],
    'auroc': []
}

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=0,
        validation_data=(X_fold_val, y_fold_val)
    )

    y_pred_probs = model.predict(X_fold_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_fold_val, axis=1)

    acc = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    auroc = roc_auc_score(y_fold_val[:, 1], y_pred_probs[:, 1])

    fold_metrics['accuracy'].append(acc)
    fold_metrics['sensitivity'].append(sens)
    fold_metrics['specificity'].append(spec)
    fold_metrics['auroc'].append(auroc)

    print(f'Fold {fold_no} Results: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}')
    fold_no += 1

print(f'\n------------------------------------------------------------------------')
print(f'Summary Results (Mean ± STD) over 10 Folds:')
for metric, values in fold_metrics.items():
    print(f'{metric.capitalize()}: {np.mean(values)*100:.2f}% ± {np.std(values)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Fold 1 Results: Acc=1.0000, Sens=1.0000, Spec=1.0000, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 2 ...
Fold 2 Results: Acc=0.9756, Sens=1.0000, Spec=0.9600, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 3 ...
Fold 3 Results: Acc=1.0000, Sens=1.0000, Spec=1.0000, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 4 ...
Fold 4 Results: Acc=0.9000, Sens=0.9130, Spec=0.8824, AUROC=0.9770
------------------------------------------------------------------------
Training for Fold 5 ...
Fold 5 Results: Acc=1.0000, Sens=1.0000, Spec=1.0000, AUROC=1.0000
------------------------------------------------------------------------
Training for Fold 6 ...
Fold 6 Results: Acc=0.8750, Se

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

y_train_labels = np.argmax(y_train_full, axis=1)
X_train_flat = X_train_full.reshape(X_train_full.shape[0], -1)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM Linear': SVC(kernel='linear', probability=True, random_state=42),
    'SVM RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
results = []

for name, model_obj in models.items():
    print(f"\n{'='*30}\nTraining {name}\n{'='*30}")
    metrics = {'acc': [], 'sens': [], 'spec': [], 'auroc': []}

    fold_idx = 1
    for train_idx, val_idx in kfold.split(X_train_flat):
        X_f_train, X_f_val = X_train_flat[train_idx], X_train_flat[val_idx]
        y_f_train, y_f_val = y_train_labels[train_idx], y_train_labels[val_idx]

        model_obj.fit(X_f_train, y_f_train)

        y_p = model_obj.predict(X_f_val)
        y_p_prob = model_obj.predict_proba(X_f_val)[:, 1]

        # Metrics calculation
        acc = accuracy_score(y_f_val, y_p)
        cm = confusion_matrix(y_f_val, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        auroc = roc_auc_score(y_f_val, y_p_prob)

        metrics['acc'].append(acc)
        metrics['sens'].append(sens)
        metrics['spec'].append(spec)
        metrics['auroc'].append(auroc)

        print(f"Fold {fold_idx}: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}")
        fold_idx += 1

    results.append({
        'Model': name,
        'Accuracy (%)': f"{np.mean(metrics['acc'])*100:.2f} ± {np.std(metrics['acc'])*100:.2f}",
        'Sensitivity (%)': f"{np.mean(metrics['sens'])*100:.2f} ± {np.std(metrics['sens'])*100:.2f}",
        'Specificity (%)': f"{np.mean(metrics['spec'])*100:.2f} ± {np.std(metrics['spec'])*100:.2f}",
        'AUROC': f"{np.mean(metrics['auroc']):.4f} ± {np.std(metrics['auroc']):.4f}"
    })

print("\nFinal Summary Table:")
display(pd.DataFrame(results))


Training Logistic Regression
Fold 1: Acc=0.8293, Sens=0.6111, Spec=1.0000, AUROC=0.8454
Fold 2: Acc=0.9024, Sens=0.8750, Spec=0.9200, AUROC=0.9475
Fold 3: Acc=0.8049, Sens=0.8500, Spec=0.7619, AUROC=0.9000
Fold 4: Acc=0.8250, Sens=0.8696, Spec=0.7647, AUROC=0.9156
Fold 5: Acc=0.8500, Sens=0.8095, Spec=0.8947, AUROC=0.9373
Fold 6: Acc=0.8750, Sens=0.8846, Spec=0.8571, AUROC=0.9258
Fold 7: Acc=0.9250, Sens=0.9091, Spec=0.9444, AUROC=0.9419
Fold 8: Acc=0.8500, Sens=0.8333, Spec=0.8636, AUROC=0.9040
Fold 9: Acc=0.7750, Sens=0.8421, Spec=0.7143, AUROC=0.8647
Fold 10: Acc=0.7750, Sens=0.7368, Spec=0.8095, AUROC=0.8195

Training SVM Linear
Fold 1: Acc=0.8049, Sens=0.6111, Spec=0.9565, AUROC=0.8720
Fold 2: Acc=0.9024, Sens=0.8750, Spec=0.9200, AUROC=0.9325
Fold 3: Acc=0.8293, Sens=0.8500, Spec=0.8095, AUROC=0.8905
Fold 4: Acc=0.8500, Sens=0.8696, Spec=0.8235, AUROC=0.9412
Fold 5: Acc=0.8500, Sens=0.8571, Spec=0.8421, AUROC=0.9298
Fold 6: Acc=0.8500, Sens=0.8462, Spec=0.8571, AUROC=0.9451
Fold

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:24:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 1: Acc=0.7805, Sens=0.7222, Spec=0.8261, AUROC=0.8768


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:24:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 2: Acc=0.8049, Sens=0.6875, Spec=0.8800, AUROC=0.9200


...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:24:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 9: Acc=0.8750, Sens=0.8421, Spec=0.9048, AUROC=0.9173


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:25:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 10: Acc=0.8000, Sens=0.8421, Spec=0.7619, AUROC=0.8672

Training MLP
Fold 1: Acc=0.7561, Sens=0.6111, Spec=0.8696, AUROC=0.9034
Fold 2: Acc=0.8537, Sens=0.7500, Spec=0.9200, AUROC=0.9725
Fold 3: Acc=0.8293, Sens=0.8500, Spec=0.8095, AUROC=0.9048
Fold 4: Acc=0.8500, Sens=0.8696, Spec=0.8235, AUROC=0.9105
Fold 5: Acc=0.9250, Sens=0.9048, Spec=0.9474, AUROC=0.9825
Fold 6: Acc=0.8750, Sens=0.8462, Spec=0.9286, AUROC=0.9643
Fold 7: Acc=0.8500, Sens=0.8636, Spec=0.8333, AUROC=0.9495
Fold 8: Acc=0.7750, Sens=0.9444, Spec=0.6364, AUROC=0.8308
Fold 9: Acc=0.7750, Sens=0.8421, Spec=0.7143, AUROC=0.8571
Fold 10: Acc=0.8000, Sens=0.8421, Spec=0.7619, AUROC=0.8271

Final Summary Table:


,Model,Accuracy (%),Sensitivity (%),Specificity (%),AUROC
0,Logistic Regression,84.12 ± 4.75,82.21 ± 8.35,85.30 ± 8.58,0.9002 ± 0.0413
1,SVM Linear,83.87 ± 4.76,81.22 ± 8.95,85.88 ± 6.04,0.9054 ± 0.0380
2,SVM RBF,79.65 ± 4.57,75.77 ± 8.15,83.25 ± 6.17,0.9019 ± 0.0301
3,Random Forest,80.40 ± 3.56,76.62 ± 8.99,83.29 ± 7.20,0.8911 ± 0.0422
4,XGBoost,81.65 ± 4.42,81.68 ± 5.80,81.29 ± 10.84,0.9014 ± 0.0352
5,MLP,82.89 ± 4.98,83.24 ± 8.75,82.44 ± 9.43,0.9102 ± 0.0545
